<a href="https://colab.research.google.com/github/usmanumer038/ml-internship-work/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/usmanumer038/ml-internship-work/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**My lane:** Predict if content will trend UP in the next 30 days,
using engagement signals from the current period.

**Task type: CLASSIFICATION (Binary)**

Why classification, not regression?
- Output: YES/NO (will it trend up or not), not a continuous number
- Decision: A creator decides whether to update/promote based on trend forecast
- Target variable: `trend_direction` (UP, DOWN, STABLE) → we predict UP vs rest

This is a binary classification problem:
- Positive class: `trend_direction == 'up'` (content is gaining momentum)
- Negative class: `trend_direction == 'down'` or `'stable'` (content is not trending)

In [9]:
import pandas as pd

print("="*60)
print("SECTION 1: VERIFY CLASSIFICATION TASK")
print("="*60)

# Check the target variable
print("\nTarget variable: trend_direction")
print(f"Unique values: {df['trend_direction'].unique()}")
print(f"Value counts:")
print(df['trend_direction'].value_counts())

print(f"\n✓ Binary classification: we predict UP vs other (DOWN/STABLE)")

SECTION 1: VERIFY CLASSIFICATION TASK

Target variable: trend_direction
Unique values: ['down' 'stable' 'new' 'up' 'flat']
Value counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

✓ Binary classification: we predict UP vs other (DOWN/STABLE)


## 2. Target or proxy

**Target:** `trend_direction == 'up'` (binary: 1 if up, 0 otherwise)

**Where does this come from?**
- Observed outcome: FlyRank already measures and labels trend direction
- Timing: This is the ACTUAL trend direction (not predicted), observed in the real data
- Why it's real: Creators see this on the platform—if content is trending up, it appears in feeds

**Why this target makes sense:**
- It's directly actionable: creator sees "your content is trending up" and decides to promote
- It's measured: trend_direction already exists in the data
- It's not a rule we invented: it's FlyRank's actual labeling of content momentum


In [12]:
print("="*60)
print("SECTION 2: TARGET VARIABLE DETAILS")
print("="*60)

print(f"\nTarget distribution:")
print(df['target'].value_counts().sort_index())

print(f"\n\nSample rows with target:")
sample_cols = ['content_id', 'trend_direction', 'engagement_rate', 'scroll_rate', 'ctr', 'target']
print(df[sample_cols].head(10))

print(f"\n✓ Target is OBSERVED (not derived from a rule)")
print(f"✓ Target is BINARY (0 or 1)")
print(f"✓ Ready to use as training label")

SECTION 2: TARGET VARIABLE DETAILS

Target distribution:
target
0    25612
1     4388
Name: count, dtype: int64


Sample rows with target:
             content_id trend_direction  engagement_rate  scroll_rate   ctr  \
0  content_304f48230142            down             5.88         4.55  0.76   
1  content_a1fb4e703a9e            down             0.00        10.00  0.05   
2  content_9aa793d4d895            down             0.00        28.57  0.09   
3  content_331d6c4de07b          stable             1.28         3.45  0.49   
4  content_d99b7a2d90ca            down             0.00        24.29  0.13   
5  content_d4084a4bc775            down             0.00        25.00  0.03   
6  content_9a34b442b552            down             0.00         0.00  0.00   
7  content_a63219c6e95a          stable             3.57         7.14  0.06   
8  content_5e6c160719bc            down             5.88         6.25  0.09   
9  content_c27558df2b0c            down             0.00         0.00  

## 3. Success metric

**Metric: ROC-AUC ≥ 0.75**

**Why ROC-AUC?**
- Our dataset is imbalanced: 14.6% trending up, 85.4% not
- Accuracy would be useless (could just predict "everything stays down")
- ROC-AUC measures: "Can the model rank trending content higher than non-trending?"
- It balances precision (avoid false alarms) and recall (catch real trends)

**Why 0.75?**
- 0.5 = random guessing
- 0.75 = good discrimination
- 0.9+ = excellent (probably overfit)

**Cost of errors:**
- False Positive (predict UP, doesn't): Creator wastes ~$50 promoting a flop
- False Negative (predict NOT-UP, but it does): Creator misses ~$500 in revenue
- Recall (catching real trends) matters more, so we need a metric that punishes misses
- ROC-AUC balances both

In [13]:
from sklearn.metrics import roc_auc_score
import numpy as np

print("="*60)
print("SECTION 3: SUCCESS METRIC VALIDATION")
print("="*60)

print(f"\nMetric: ROC-AUC")
print(f"Target: ≥ 0.75")

# Baseline check: random predictions
random_predictions = np.random.rand(len(df))
baseline_auc = roc_auc_score(df['target'], random_predictions)

print(f"\nBaseline AUC (random guessing): {baseline_auc:.3f}")
print(f"Our target: 0.75")
print(f"Good model would be: 0.15 points above baseline")

print(f"\n✓ Metric is measurable")
print(f"✓ Metric rewards finding actual UP cases")
print(f"✓ Ready for model training")

SECTION 3: SUCCESS METRIC VALIDATION

Metric: ROC-AUC
Target: ≥ 0.75

Baseline AUC (random guessing): 0.493
Our target: 0.75
Good model would be: 0.15 points above baseline

✓ Metric is measurable
✓ Metric rewards finding actual UP cases
✓ Ready for model training


## 4. The unit of analysis, as a real dataframe

**One row = ONE CONTENT ITEM** (a piece of content on FlyRank)

Each row represents:
- A single piece of content (blog post, article, etc.)
- Its performance metrics over a 90-day trailing window
- Its current engagement signals (ctr, engagement_rate, scroll_rate)
- Its trend direction (observed label)

**What we predict for each row:**
- Will this content trend UP in the next 30 days?

**Grain:** One row per content_id, one observation per content item

In [14]:
print("="*60)
print("SECTION 4: UNIT OF ANALYSIS")
print("="*60)

print(f"\nUnit of analysis: ONE CONTENT ITEM (one row = one piece of content)")
print(f"Total rows: {len(df):,}")
print(f"Total unique content items: {df['content_id'].nunique():,}")

print(f"\n\nKey columns for prediction:")
key_cols = ['content_id', 'client_id', 'engagement_rate', 'scroll_rate', 'ctr',
            'impressions_90d', 'clicks_90d', 'pageviews_90d', 'word_count',
            'trend_direction', 'target']

print(df[key_cols].head(10))

print(f"\n\nColumn descriptions:")
print(f"- content_id: unique identifier for this piece of content")
print(f"- client_id: which FlyRank customer owns this content")
print(f"- engagement_rate: % of impressions that led to engagement (0-100)")
print(f"- scroll_rate: % of sessions with scroll events (0-100)")
print(f"- ctr: click-through rate from search (0-100, rate ×100)")
print(f"- impressions_90d: total impressions in last 90 days")
print(f"- clicks_90d: total clicks in last 90 days")
print(f"- pageviews_90d: total pageviews in last 90 days")
print(f"- word_count: length of content in words")
print(f"- trend_direction: OBSERVED trend (up/down/stable/new/flat)")
print(f"- target: our binary label (1 if trend='up', 0 otherwise)")

print(f"\n✓ Unit of analysis is clear: one content item per row")
print(f"✓ Features are observable at prediction time")
print(f"✓ Target is observed outcome (trend_direction)")

SECTION 4: UNIT OF ANALYSIS

Unit of analysis: ONE CONTENT ITEM (one row = one piece of content)
Total rows: 30,000
Total unique content items: 30,000


Key columns for prediction:
             content_id          client_id  engagement_rate  scroll_rate  \
0  content_304f48230142  client_f369cb89fc             5.88         4.55   
1  content_a1fb4e703a9e  client_4e07408562             0.00        10.00   
2  content_9aa793d4d895  client_7f2253d7e2             0.00        28.57   
3  content_331d6c4de07b  client_19581e27de             1.28         3.45   
4  content_d99b7a2d90ca  client_3fdba35f04             0.00        24.29   
5  content_d4084a4bc775  client_f369cb89fc             0.00        25.00   
6  content_9a34b442b552  client_8722616204             0.00         0.00   
7  content_a63219c6e95a  client_19581e27de             3.57         7.14   
8  content_5e6c160719bc  client_6208ef0f77             5.88         6.25   
9  content_c27558df2b0c  client_19581e27de             0.00

## 5. Why ML beats a fixed rule here

**A simple rule would fail because engagement signals are tangled.**

Example of a failing rule:
- Rule: "If engagement_rate > 5, predict UP"
- Problem: Look at row 0: engagement_rate = 5.88, but trend_direction = DOWN
- The rule predicts UP, but it's wrong

Another failing rule:
- Rule: "If scroll_rate > 10, predict UP"
- Problem: Row 1 has scroll_rate = 10.00 but trend = DOWN
- Again, the rule fails

**Why multiple signals matter:**
- engagement_rate alone doesn't predict trend
- scroll_rate alone doesn't predict trend
- ctr alone doesn't predict trend
- But the COMBINATION might: high engagement + high scroll + high ctr = trending up

**Why ML is necessary:**
- ML learns which combinations of signals matter
- ML learns the weights: maybe scroll_rate matters 3x more than ctr
- ML learns non-linear patterns: maybe "high scroll BUT low ctr" means something different than "high scroll AND high ctr"
- ML adapts: if the pattern shifts over time (scroll matters less, impressions matter more), the model can retrain

**A fixed rule can't do any of this.** It's static. ML learns from data.

In [15]:
print("="*60)
print("SECTION 5: WHY ML BEATS A FIXED RULE")
print("="*60)

print("\n\nTesting simple rules:")
print("="*50)

# Rule 1: If engagement_rate > 5, predict UP
rule1_pred = (df['engagement_rate'] > 5).astype(int)
rule1_correct = (rule1_pred == df['target']).sum()
rule1_acc = rule1_correct / len(df)

print(f"\nRule 1: 'If engagement_rate > 5, predict UP'")
print(f"Accuracy: {rule1_acc:.3f}")
print(f"But look at actual data:")
print(df[['engagement_rate', 'trend_direction', 'target']].head(10))

# Rule 2: If scroll_rate > 10, predict UP
rule2_pred = (df['scroll_rate'] > 10).astype(int)
rule2_correct = (rule2_pred == df['target']).sum()
rule2_acc = rule2_correct / len(df)

print(f"\n\nRule 2: 'If scroll_rate > 10, predict UP'")
print(f"Accuracy: {rule2_acc:.3f}")

# Rule 3: Combination (engagement > 5 AND scroll > 10)
rule3_pred = ((df['engagement_rate'] > 5) & (df['scroll_rate'] > 10)).astype(int)
rule3_correct = (rule3_pred == df['target']).sum()
rule3_acc = rule3_correct / len(df)

print(f"\n\nRule 3: 'If engagement_rate > 5 AND scroll_rate > 10, predict UP'")
print(f"Accuracy: {rule3_acc:.3f}")

# Baseline: predict everything is NOT trending up
baseline_pred = np.zeros(len(df))
baseline_acc = (baseline_pred == df['target']).sum() / len(df)

print(f"\n\nBaseline: 'Predict everything is NOT trending up'")
print(f"Accuracy: {baseline_acc:.3f}")

print(f"\n\n{'='*50}")
print(f"Summary:")
print(f"{'='*50}")
print(f"Rule 1 (engagement > 5): {rule1_acc:.3f}")
print(f"Rule 2 (scroll > 10): {rule2_acc:.3f}")
print(f"Rule 3 (combination): {rule3_acc:.3f}")
print(f"Baseline (predict all NOT-UP): {baseline_acc:.3f}")
print(f"\n✓ All rules fail or barely beat baseline")
print(f"✓ ML can learn better patterns by combining signals intelligently")
print(f"✓ This is why we need ML, not just rules")

SECTION 5: WHY ML BEATS A FIXED RULE


Testing simple rules:

Rule 1: 'If engagement_rate > 5, predict UP'
Accuracy: 0.767
But look at actual data:
   engagement_rate trend_direction  target
0             5.88            down       0
1             0.00            down       0
2             0.00            down       0
3             1.28          stable       0
4             0.00            down       0
5             0.00            down       0
6             0.00            down       0
7             3.57          stable       0
8             5.88            down       0
9             0.00            down       0


Rule 2: 'If scroll_rate > 10, predict UP'
Accuracy: 0.569


Rule 3: 'If engagement_rate > 5 AND scroll_rate > 10, predict UP'
Accuracy: 0.794


Baseline: 'Predict everything is NOT trending up'
Accuracy: 0.854


Summary:
Rule 1 (engagement > 5): 0.767
Rule 2 (scroll > 10): 0.569
Rule 3 (combination): 0.794
Baseline (predict all NOT-UP): 0.854

✓ All rules fail or barely beat

## 6. Self-check

Before submitting, I confirm:

✅ **Task type is named clearly:** CLASSIFICATION (Binary)
   - Predicting: Is trend_direction == 'up'? (Yes/No)

✅ **Target is observed, not invented:** trend_direction is real data from FlyRank
   - Not derived from a rule we made up
   - Measured in the actual dataset

✅ **Success metric is one number:** ROC-AUC ≥ 0.75
   - Handles class imbalance (14.6% positive)
   - Balances precision and recall
   - Measurable before training

✅ **Unit of analysis is clear:** One row = one content item
   - 30,000 unique content pieces
   - Each has engagement signals (ctr, scroll_rate, engagement_rate, etc.)
   - Each has an observed trend direction label

✅ **ML is necessary:** Simple rules fail
   - Rule 1 (engagement > 5): 76.7% accuracy
   - Rule 3 (combination): 79.4% accuracy
   - Baseline (predict all NOT-UP): 85.4% accuracy
   - No single rule or simple combination beats baseline
   - ML can learn complex interactions between signals

✅ **Action is real:** Creators use this to decide whether to promote/update content
   - High predicted probability → invest in promotion
   - Low predicted probability → skip or deprioritize

**This notebook is ready to submit.**

In [16]:
print("="*60)
print("SECTION 6: FINAL SELF-CHECK")
print("="*60)

checks = {
    "Task type named": "✅ CLASSIFICATION (Binary)",
    "Target is observed": f"✅ trend_direction (real data, {df['target'].sum():,} positive cases)",
    "Success metric defined": "✅ ROC-AUC ≥ 0.75",
    "Unit of analysis clear": f"✅ One row = one content item ({len(df):,} total)",
    "ML beats fixed rule": "✅ Best rule: 79.4%, Baseline: 85.4%, Need ML to do better",
    "Action is real": "✅ Creator decides to promote based on prediction"
}

for check, status in checks.items():
    print(f"{status}")
    print(f"  {check}")

print(f"\n{'='*60}")
print(f"✅ NOTEBOOK IS COMPLETE AND READY TO SUBMIT")
print(f"{'='*60}")

SECTION 6: FINAL SELF-CHECK
✅ CLASSIFICATION (Binary)
  Task type named
✅ trend_direction (real data, 4,388 positive cases)
  Target is observed
✅ ROC-AUC ≥ 0.75
  Success metric defined
✅ One row = one content item (30,000 total)
  Unit of analysis clear
✅ Best rule: 79.4%, Baseline: 85.4%, Need ML to do better
  ML beats fixed rule
✅ Creator decides to promote based on prediction
  Action is real

✅ NOTEBOOK IS COMPLETE AND READY TO SUBMIT


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.